# 01 — Dataset Curation

**Purpose.** Reproduce the 234-sample PHP benchmark used in this study by curating samples from the NIST Software Assurance Reference Dataset (SARD) PHP Vulnerability Test Suite. The output is a directory containing exactly 100 CWE-89 (SQL Injection), 100 CWE-78 (OS Command Injection), and 34 Safe samples.

**Inputs.** None (the raw corpus is downloaded by this notebook).

**Outputs.**
- `<BASE_DIR>/raw_sard_php/` — the full SARD PHP Vulnerability Test Suite (~42,000 files), cloned from GitHub.
- `<BASE_DIR>/final_dataset/CWE_89_SQLi/` — 100 SQL Injection samples.
- `<BASE_DIR>/final_dataset/CWE_78_CmdInj/` — 100 OS Command Injection samples.
- `<BASE_DIR>/final_dataset/Safe_Code/` — 34 safe samples.

**Citation of source corpus.** Stivalet, B. and Fong, E. (2016). Large Scale Generation of Complex and Faulty PHP Test Cases. *IEEE 9th International Conference on Software Testing, Verification and Validation (ICST)*. The corpus is hosted at https://github.com/stivalet/PHP-Vulnerability-test-suite.

**Note on CWE-79 (XSS).** XSS is not represented in the final dataset because the curation pipeline applied to the SARD CWE-79 stratum (using filename-pattern extraction) yielded no candidate samples suitable for source-to-sink evaluation. See Section 7 of this notebook and Section 3.1.3 of the paper for details.


## 1. Setup

Edit `BASE_DIR` below to point to a writable directory on your machine. All other paths are derived from `BASE_DIR`.

In [ ]:
import os
import re
import shutil
import subprocess
from pathlib import Path

# ---- USER-EDITABLE ----
# Set BASE_DIR_OVERRIDE to a writable directory if you want to override the default.
# By default, BASE_DIR is auto-located at <repo_root>/workspace, which works whether
# you launch Jupyter from the repository root or from notebooks/.
# On Google Colab, set this to '/content/drive/MyDrive/llm-vuln-detection-ablation'.
BASE_DIR_OVERRIDE = None  # e.g., Path('/content/drive/MyDrive/llm-vuln-detection-ablation')
# -----------------------

def find_repo_root() -> Path:
    """Locate the repository root by walking upward from the current working directory
    in search of a sentinel file (README.md, requirements.txt, or .git directory).
    Falls back to the current working directory if no sentinel is found.
    """
    sentinels = ('README.md', 'requirements.txt', '.git')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if any((candidate / s).exists() for s in sentinels):
            return candidate
    return cwd

REPO_ROOT = find_repo_root()
BASE_DIR  = Path(BASE_DIR_OVERRIDE).resolve() if BASE_DIR_OVERRIDE else (REPO_ROOT / 'workspace')

# Derived paths.
RAW_DIR              = BASE_DIR / 'raw_sard_php'
FINAL_DATASET_DIR    = BASE_DIR / 'final_dataset'
OUT_SQLI_DIR         = FINAL_DATASET_DIR / 'CWE_89_SQLi'
OUT_CMDINJ_DIR       = FINAL_DATASET_DIR / 'CWE_78_CmdInj'
OUT_SAFE_DIR         = FINAL_DATASET_DIR / 'Safe_Code'

# Canonical file list (committed under <repo_root>/data/).
CANONICAL_FILE_LIST  = REPO_ROOT / 'data' / 'file_list_234.txt'

# Public source corpus.
SARD_REPO_URL = 'https://github.com/stivalet/PHP-Vulnerability-test-suite.git'

# Create base directories if they do not exist.
BASE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
for d in (OUT_SQLI_DIR, OUT_CMDINJ_DIR, OUT_SAFE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT             : {REPO_ROOT}')
print(f'BASE_DIR              : {BASE_DIR}')
print(f'RAW_DIR               : {RAW_DIR}')
print(f'FINAL_DATASET_DIR     : {FINAL_DATASET_DIR}')
print(f'CANONICAL_FILE_LIST   : {CANONICAL_FILE_LIST}')
print(f'CANONICAL_FILE_LIST exists: {CANONICAL_FILE_LIST.exists()}')


## 2. Download raw SARD PHP corpus

Clone the PHP Vulnerability Test Suite from GitHub into `RAW_DIR`. If `RAW_DIR` already exists and contains a clone, this step is skipped to avoid redundant downloads.

Expected size: ~7 MB, ~42,000 files. Typical clone time: 30–60 seconds on a typical broadband connection.

In [ ]:
def is_existing_clone(path: Path) -> bool:
    """Return True if `path` already contains a SARD PHP corpus clone."""
    if not path.exists():
        return False
    # Heuristic: a successful clone has a .git dir and at least 1000 .php files.
    if not (path / '.git').exists():
        return False
    php_count = sum(1 for _ in path.rglob('*.php'))
    return php_count >= 1000

if is_existing_clone(RAW_DIR):
    php_count = sum(1 for _ in RAW_DIR.rglob('*.php'))
    print(f'Existing clone detected at {RAW_DIR} ({php_count} .php files). Skipping download.')
else:
    if RAW_DIR.exists():
        # Path exists but is not a valid clone; remove and re-clone.
        shutil.rmtree(RAW_DIR)
    RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f'Cloning {SARD_REPO_URL} into {RAW_DIR} ...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', SARD_REPO_URL, str(RAW_DIR)],
        check=True,
    )
    php_count = sum(1 for _ in RAW_DIR.rglob('*.php'))
    print(f'Clone complete. {php_count} .php files in {RAW_DIR}.')


## 3. Curate CWE-89 (SQL Injection) samples

We extract candidate samples whose filename begins with the prefix `CWE_89__` (corresponding to the SARD CWE-89 generation routines), then select the specific 100 samples that match `data/file_list_234.txt`. The file list — committed to this repository — is the canonical specification of dataset composition.

Target: exactly 100 samples.

In [ ]:
def find_files_by_prefix(root: Path, prefix: str) -> list:
    """Return all .php files under `root` whose basename starts with `prefix`.
    Results are returned in deterministic (sorted) order to support reproducibility.
    """
    return sorted(p for p in root.rglob('*.php') if p.name.startswith(prefix))

TARGET_N_SQLI = 100

# Locate all CWE-89 candidates in the raw corpus.
sqli_candidates = find_files_by_prefix(RAW_DIR, 'CWE_89__')
print(f'CWE-89 candidates found: {len(sqli_candidates)}')

# Read the canonical file list to select exactly the 100 samples used in the paper.
with open(CANONICAL_FILE_LIST) as f:
    canonical_basenames_sqli = {
        Path(line.strip()).name
        for line in f
        if line.strip().startswith('CWE_89_SQLi/')
    }
print(f'Canonical CWE-89 basenames in file_list_234.txt: {len(canonical_basenames_sqli)}')

selected_sqli = [p for p in sqli_candidates if p.name in canonical_basenames_sqli]
print(f'Selected (intersection of candidates and canonical list): {len(selected_sqli)}')

if len(selected_sqli) != TARGET_N_SQLI:
    raise RuntimeError(
        f'Expected {TARGET_N_SQLI} CWE-89 samples but selected {len(selected_sqli)}. '
        f'Verify that RAW_DIR points to a complete SARD PHP corpus clone.'
    )

# Copy to output directory.
for src in selected_sqli:
    shutil.copy2(src, OUT_SQLI_DIR / src.name)

print(f'Copied {len(selected_sqli)} CWE-89 samples to {OUT_SQLI_DIR}')


## 4. Curate CWE-78 (OS Command Injection) samples

Same procedure as Section 3, but for OS Command Injection samples (filename prefix `CWE_78__`). The 100 selected CWE-78 samples include both `GET`-source and `POST`-source variants, as documented by the canonical file list.

Target: exactly 100 samples.

In [ ]:
TARGET_N_CMDINJ = 100

cmdinj_candidates = find_files_by_prefix(RAW_DIR, 'CWE_78__')
print(f'CWE-78 candidates found: {len(cmdinj_candidates)}')

with open(CANONICAL_FILE_LIST) as f:
    canonical_basenames_cmdinj = {
        Path(line.strip()).name
        for line in f
        if line.strip().startswith('CWE_78_CmdInj/')
    }
print(f'Canonical CWE-78 basenames in file_list_234.txt: {len(canonical_basenames_cmdinj)}')

selected_cmdinj = [p for p in cmdinj_candidates if p.name in canonical_basenames_cmdinj]
print(f'Selected (intersection of candidates and canonical list): {len(selected_cmdinj)}')

if len(selected_cmdinj) != TARGET_N_CMDINJ:
    raise RuntimeError(
        f'Expected {TARGET_N_CMDINJ} CWE-78 samples but selected {len(selected_cmdinj)}. '
        f'Verify that RAW_DIR points to a complete SARD PHP corpus clone.'
    )

for src in selected_cmdinj:
    shutil.copy2(src, OUT_CMDINJ_DIR / src.name)

print(f'Copied {len(selected_cmdinj)} CWE-78 samples to {OUT_CMDINJ_DIR}')


## 5. Curate Safe samples

The 34 Safe samples in this study are drawn from the SARD CWE-862 (Missing Authorization) stratum, specifically the patched/safe variants whose data flow does not reach a vulnerable sink. They are used as negative controls for the binary Vulnerable/Safe classification task.

Target: exactly 34 samples.

In [ ]:
TARGET_N_SAFE = 34

# Locate Safe candidates in the raw corpus (CWE-862 access-control variants).
safe_candidates = find_files_by_prefix(RAW_DIR, 'CWE_862_')
print(f'Safe candidates (CWE_862_*) found: {len(safe_candidates)}')

with open(CANONICAL_FILE_LIST) as f:
    canonical_basenames_safe = {
        Path(line.strip()).name
        for line in f
        if line.strip().startswith('Safe_Code/')
    }
print(f'Canonical Safe basenames in file_list_234.txt: {len(canonical_basenames_safe)}')

selected_safe = [p for p in safe_candidates if p.name in canonical_basenames_safe]
print(f'Selected (intersection of candidates and canonical list): {len(selected_safe)}')

if len(selected_safe) != TARGET_N_SAFE:
    raise RuntimeError(
        f'Expected {TARGET_N_SAFE} Safe samples but selected {len(selected_safe)}. '
        f'Verify that RAW_DIR points to a complete SARD PHP corpus clone.'
    )

for src in selected_safe:
    shutil.copy2(src, OUT_SAFE_DIR / src.name)

print(f'Copied {len(selected_safe)} Safe samples to {OUT_SAFE_DIR}')


## 6. Verify against canonical file list

Confirm that the curated dataset under `final_dataset/` contains exactly the 234 files specified in `data/file_list_234.txt`. This is a strict equality check: any deviation (missing file, extra file, wrong category) will raise an `AssertionError`. Passing this check guarantees that downstream notebooks (02–05) operate on the same dataset as the paper.

In [ ]:
# Read canonical list as relative paths (e.g., 'CWE_89_SQLi/<filename>.php').
with open(CANONICAL_FILE_LIST) as f:
    canonical_relpaths = sorted(line.strip() for line in f if line.strip())

# Enumerate actual files under final_dataset/ as relative paths.
actual_relpaths = sorted(
    str(p.relative_to(FINAL_DATASET_DIR)).replace(os.sep, '/')
    for p in FINAL_DATASET_DIR.rglob('*.php')
)

missing = set(canonical_relpaths) - set(actual_relpaths)
extra   = set(actual_relpaths)    - set(canonical_relpaths)

print(f'Canonical files: {len(canonical_relpaths)}')
print(f'Actual files   : {len(actual_relpaths)}')
print(f'Missing        : {len(missing)}')
print(f'Extra          : {len(extra)}')

if missing:
    print('First missing files (up to 10):')
    for f in sorted(missing)[:10]:
        print(f'  - {f}')
if extra:
    print('First extra files (up to 10):')
    for f in sorted(extra)[:10]:
        print(f'  - {f}')

assert not missing and not extra, (
    f'Curated dataset does not match canonical list '
    f'(missing={len(missing)}, extra={len(extra)}).'
)

print('\nVerification PASSED: curated dataset exactly matches data/file_list_234.txt.')


## 7. Note on CWE-79 (XSS) absence

Cross-Site Scripting (CWE-79) is not represented in the curated dataset. We applied the same filename-pattern extraction procedure (Section 3) to the SARD CWE-79 stratum: the candidate prefix `CWE_79__GET__` did locate XSS-related samples in the raw corpus, but none of these samples were suitable for the source-to-sink evaluation paradigm used in this study. Specifically, XSS sinks are output-rendering contexts (HTML attribute interpolation, JavaScript embedding, etc.) rather than the query-execution sinks (`mysqli_query`, `system`, `exec`, etc.) that match the source-to-sink taint analysis prompts of Variants C and E.

We therefore exclude XSS from the empirical scope of this study. This scope decision is discussed as a threat to validity and a future-work direction in Section 5.4 of the paper. An audit log confirming the absence of suitable XSS candidates is provided in `docs/cwe79_absence_evidence.txt`.
